<a href="https://colab.research.google.com/github/aryanks692/Flyrank-ai-Notebook/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [2]:

#Finding 1

 #A finding about content refreshes and performance.

#Write:

#Finding: The paper reports that pages that received content refreshes showed stronger subsequent performance than the comparison group.

#Methodology question — where does the label come from?
#I would want to understand how a page was labeled as "refreshed." Was the refresh identified using a recorded content change, a specific threshold of change, or another signal? I would also want to confirm that the refresh classification was determined independently of the later performance outcome.

#Methodology question — does the validation design support the claim?
#The comparison can support an observed association if the refreshed and comparison groups and their measurement windows are clearly defined. However, pages selected for refreshing may have been different from the beginning. Therefore, I would be cautious about interpreting the result as proof that refreshing caused the performance improvement. The finding is more appropriately described as an observed or directional relationship unless the study design provides stronger causal evidence.
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

#Finding: The paper reports a relationship between content characteristics and search performance.

#Methodology question — where does the label come from?
#I would want to understand exactly how the search-performance outcome was defined and which time period was used to calculate it. In particular, I would check that the content characteristics were measured before the performance outcome rather than using information from the outcome period.

#Methodology question — does the validation design support the claim?
#The observed relationship can be useful for directional decision-support, but I would want to know whether the validation design accounts for differences between clients, content types, content age, and existing search visibility. If related pages from the same clients appear in both development and evaluation data, the measured relationship may not generalize to unseen clients.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [4]:
!git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 283, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 283 (delta 112), reused 83 (delta 83), pack-reused 138 (from 1)
Receiving objects: 100% (283/283), 1.85 MiB | 5.50 MiB/s, done.
Resolving deltas: 100% (153/153), done.


In [5]:
%cd flyrank-ml-internship-starter

/content/flyrank-ml-internship-starter


In [6]:
!ls

AGENTS.md  DATA_USE.md	LICENSE    README.md	     SETUP.md	 work
CLAUDE.md  docs		notebooks  requirements.txt  skills
data	   GUIDE.md	outputs    scripts	     submission


In [7]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


##Target

In [9]:
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

In [10]:
print(df["is_declining_label"].value_counts())

is_declining_label
1    16262
0    13738
Name: count, dtype: int64


In [11]:
base_rate = df["is_declining_label"].mean()

print("Declining base rate:", base_rate)

Declining base rate: 0.5420666666666667


#Week-5 features

In [16]:
feature_cols = [
    "impressions_90d",
    "sessions_90d",
    "content_age_days",
    ...
]

In [14]:
print(feature_cols)

['impressions_90d', 'sessions_90d', 'content_age_days', Ellipsis]


In [18]:
feature_cols = [
    "impressions_90d",
    "sessions_90d",
    "content_age_days"
]

X = df[feature_cols]
y = df["is_declining_label"]
groups = df["client_id"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Number of clients:", groups.nunique())

X shape: (30000, 3)
y shape: (30000,)
Number of clients: 32


In [19]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

print("Training rows:", len(train_idx))
print("Testing rows:", len(test_idx))

Training rows: 23837
Testing rows: 6163


In [20]:
train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

print("Training clients:", len(train_clients))
print("Testing clients:", len(test_clients))
print("Overlapping clients:", len(train_clients & test_clients))

Training clients: 25
Testing clients: 7
Overlapping clients: 0


#Train the model

In [22]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(
    X.iloc[train_idx],
    y.iloc[train_idx]
)

LogisticRegression(max_iter=1000, random_state=42)

#Predictions

In [23]:
y_prob = model.predict_proba(
    X.iloc[test_idx]
)[:, 1]

In [24]:
print(y_prob[:10])

[0.5984603  0.40821482 0.6284028  0.5632078  0.60048384 0.66306024
 0.57032589 0.36803786 0.60563572 0.51634073]


Calculate ROC-AUC

In [25]:
from sklearn.metrics import roc_auc_score

grouped_roc_auc = roc_auc_score(
    y.iloc[test_idx],
    y_prob
)

print("Grouped ROC-AUC:", grouped_roc_auc)

Grouped ROC-AUC: 0.5527344289157216


##Calculate Average Precision

In [26]:
from sklearn.metrics import average_precision_score

grouped_ap = average_precision_score(
    y.iloc[test_idx],
    y_prob
)

print("Grouped Average Precision:", grouped_ap)

Grouped Average Precision: 0.546146253832797


##Calculate Precision@50

In [27]:
test_results = pd.DataFrame({
    "actual": y.iloc[test_idx].values,
    "score": y_prob
})

top50 = test_results.sort_values(
    "score",
    ascending=False
).head(50)

grouped_p50 = top50["actual"].mean()

print("Grouped Precision@50:", grouped_p50)

Grouped Precision@50: 0.46


In [31]:

comparison = pd.DataFrame([
    {
        "Evaluation": "Week-5 validation",
        "ROC-AUC": 0.675,
        "Average Precision": 0.568,
        "Precision@50": 0.78
    },
    {
        "Evaluation": "Grouped client validation",
        "ROC-AUC": grouped_roc_auc,
        "Average Precision": grouped_ap,
        "Precision@50": grouped_p50
    }
])

comparison
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


,Evaluation,ROC-AUC,Average Precision,Precision@50
0,Week-5 validation,0.675000,0.568000,0.78
1,Grouped client validation,0.552734,0.546146,0.46


##Before/after interpretation

I compared the Week-5 validation result with a grouped client validation split. In the grouped split, no client appears in both the training and testing sets, so the test set represents clients that were not seen during training.

The grouped evaluation provides a more conservative estimate of generalization to unseen clients. The difference between the Week-5 and grouped results is itself informative because it shows how sensitive the measured performance is to the validation design. I therefore use the grouped-client result as the primary evidence for the model's performance.

## 3. Leakage Audit

The target `is_declining_label` is derived from `trend_direction`, where pages with `trend_direction == "down"` are assigned the positive label.

I therefore need to ensure that the model does not use `trend_direction`, `trend_pct`, or any other feature that directly or indirectly contains the target information.

In [32]:
# This cell is for CODE (numbers, a query, a check).
suspect_features = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

for feature in suspect_features:
    print(
        feature,
        "->",
        feature in feature_cols
    )
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


trend_direction -> False
trend_pct -> False
is_declining_label -> False


##Check_ID

In [33]:
id_features = [
    "content_id",
    "client_id"
]

for feature in id_features:
    print(
        feature,
        "->",
        feature in feature_cols
    )

content_id -> False
client_id -> False


##Deliberately add a leaky feature

In [34]:
df_leaky = df.copy()

df_leaky["leaky_feature"] = (
    df_leaky["trend_direction"] == "down"
).astype(int)

##Train a model with the leaky feature

In [35]:
X_leaky = df_leaky[
    feature_cols + ["leaky_feature"]
]

leaky_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

leaky_model.fit(
    X_leaky.iloc[train_idx],
    y.iloc[train_idx]
)

LogisticRegression(max_iter=1000, random_state=42)

predict

In [36]:
leaky_prob = leaky_model.predict_proba(
    X_leaky.iloc[test_idx]
)[:, 1]

##Calculate ROC_LOC

In [37]:
leaky_auc = roc_auc_score(
    y.iloc[test_idx],
    leaky_prob
)

print("Honest ROC-AUC:", grouped_roc_auc)
print("Leaky ROC-AUC:", leaky_auc)

Honest ROC-AUC: 0.5527344289157216
Leaky ROC-AUC: 1.0000000000000002


##Create the leakage comparison

In [38]:
leakage_comparison = pd.DataFrame([
    {
        "Feature set": "Honest features",
        "ROC-AUC": grouped_roc_auc
    },
    {
        "Feature set": "Honest + deliberately leaky feature",
        "ROC-AUC": leaky_auc
    }
])

leakage_comparison

,Feature set,ROC-AUC
0,Honest features,0.552734
1,Honest + deliberately leaky feature,1.000000


##Explain The experiment

### Label-derived leakage test

I deliberately introduced a feature derived directly from `trend_direction`, which is used to construct the target `is_declining_label`.

The resulting ROC-AUC increased substantially compared with the honest feature set. This confirms that the evaluation would expose obvious label-derived leakage.

The deliberately leaky feature was used only as a diagnostic experiment and is not included in the reported model. `trend_direction` and `trend_pct` remain excluded from the production feature set because they contain information used to define the target.

### Future / overlapping-window audit

I reviewed the temporal meaning of the features used by the model. A feature should only be used if the information would have been available at the prediction time.

The target is based on the observed trend direction, so features containing information from the target/outcome window could introduce leakage.

For the current model, I use `impressions_90d`, `sessions_90d`, and `content_age_days`. I do not use `trend_direction` or `trend_pct`, because these are directly involved in defining the target.

The temporal alignment of each feature should be interpreted according to the dataset's data dictionary rather than assuming that a column is safe solely because its name contains a historical window.

### Decision-derived feature audit

I checked the feature set for existing system scores, flags, recommendations, or other variables that could represent a previous decision.

No known product decision flags or existing-model prediction scores are included in the current feature set.

`client_id` is used only for grouped validation and is not used as a predictive feature.


## Feature sanity check

In [39]:
importance = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": model.coef_[0]
})

importance["absolute_coefficient"] = (
    importance["coefficient"].abs()
)

importance.sort_values(
    "absolute_coefficient",
    ascending=False
).head(10)

,feature,coefficient,absolute_coefficient
2,content_age_days,-2.994114e-03,2.994114e-03
1,sessions_90d,-5.488423e-04,5.488423e-04
0,impressions_90d,-2.008576e-07,2.008576e-07


#Error analysis

##test-results DataFrame

In [41]:
error_df = df.iloc[test_idx].copy()

error_df["actual"] = y.iloc[test_idx].values
error_df["predicted_probability"] = y_prob
error_df["predicted_label"] = (
    error_df["predicted_probability"] >= 0.5
).astype(int)

print(error_df.shape)

(6163, 48)


##False Positive

In [42]:
false_positives = error_df[
    (error_df["actual"] == 0) &
    (error_df["predicted_label"] == 1)
]

print("False positives:", len(false_positives))

False positives: 1600


In [43]:
false_positives[
    [
        "content_id",
        "client_id",
        "impressions_90d",
        "sessions_90d",
        "content_age_days",
        "actual",
        "predicted_probability"
    ]
].head(10)

,content_id,client_id,impressions_90d,sessions_90d,content_age_days,actual,predicted_probability
13,content_a5a2fbc76336,client_8527a891e2,307,4,238,0,0.563208
26,content_72c5c2d73e5a,client_4e07408562,2426,9,300,0,0.516341
36,content_bce275871a25,client_f369cb89fc,371,5,187,0,0.600207
56,content_dcebfd222b10,client_f369cb89fc,16,2,145,0,0.630369
64,content_685de0e3b7cb,client_f369cb89fc,2639,6,106,0,0.656525
78,content_dea0d86223f3,client_8527a891e2,59,3,174,0,0.609786
126,content_be5e23c0a35e,client_f369cb89fc,82,2,126,0,0.643520
135,content_670746e86425,client_4e07408562,19802,102,280,0,0.517677
143,content_fe49ee6b87c4,client_f369cb89fc,5,3,126,0,0.643398
174,content_ed4f3ec14a68,client_f369cb89fc,2,1,131,0,0.640208


#False negatives

In [44]:
false_negatives = error_df[
    (error_df["actual"] == 1) &
    (error_df["predicted_label"] == 0)
]

print("False negatives:", len(false_negatives))

False negatives: 1300


In [45]:
false_negatives[
    [
        "content_id",
        "client_id",
        "impressions_90d",
        "sessions_90d",
        "content_age_days",
        "actual",
        "predicted_probability"
    ]
].head(10)

,content_id,client_id,impressions_90d,sessions_90d,content_age_days,actual,predicted_probability
1,content_a1fb4e703a9e,client_4e07408562,15320,9,445,1,0.408215
23,content_2da6ae9d0882,client_e629fa6598,297,12,502,1,0.368038
39,content_4595e8704e07,client_8527a891e2,4,1,348,1,0.481645
54,content_ff8ea1364b59,client_e629fa6598,170,1,502,1,0.369449
58,content_caff51984338,client_e629fa6598,71,2,494,1,0.374922
81,content_16788821b64a,client_e629fa6598,320,5,502,1,0.368931
90,content_abfa53fe7911,client_e629fa6598,261,5,460,1,0.398662
94,content_9983d31c53cb,client_4e07408562,7737,7,326,1,0.496893
127,content_02141810795a,client_4e07408562,1912,1,487,1,0.379889
129,content_b4170c25efd2,client_4e07408562,2159,3,421,1,0.427134


##Look at the most uncertain mistakes

In [46]:
error_df["distance_from_0_5"] = (
    error_df["predicted_probability"] - 0.5
).abs()

In [47]:
uncertain_cases = error_df.sort_values(
    "distance_from_0_5"
).head(10)

uncertain_cases[
    [
        "content_id",
        "client_id",
        "impressions_90d",
        "sessions_90d",
        "content_age_days",
        "actual",
        "predicted_probability"
    ]
]

,content_id,client_id,impressions_90d,sessions_90d,content_age_days,actual,predicted_probability
28896,content_f924f7e00676,client_4e07408562,638,1,326,0,0.498072
25431,content_6b91347ea965,client_4e07408562,1194,1,326,1,0.498044
15012,content_42cb3441f18d,client_4e07408562,2274,1,326,1,0.497990
10745,content_8179b32b194b,client_4e07408562,1211,2,326,0,0.497906
13452,content_b8a7d1626856,client_4e07408562,1631,2,326,1,0.497885
19515,content_9ae5157e9941,client_4e07408562,2265,2,326,1,0.497853
7272,content_f41264b82587,client_8527a891e2,82,64,309,1,0.502181
26946,content_0073196442b4,client_4e07408562,65343,471,230,0,0.502193
21426,content_1aae7c2983fc,client_4e07408562,1784,3,326,1,0.497740
1756,content_13cfec0b5b2f,client_4e07408562,2208,3,326,1,0.497719


### Error analysis

I inspected false-positive and false-negative examples from the grouped-client test set.

False positives are pages that the model prioritized as potentially declining but that did not receive the declining label. False negatives are pages that received the declining label but were not prioritized by the model.

These examples show that the model does not provide a definitive classification of page health. Instead, its predictions should be interpreted as a directional ranking signal that can support further human review.

The errors may also indicate that the current feature set does not capture all factors associated with the observed declining label.

In [48]:
print("False positives by client:")
print(false_positives["client_id"].value_counts().head())

print("\nFalse negatives by client:")
print(false_negatives["client_id"].value_counts().head())

False positives by client:
client_id
client_f369cb89fc    716
client_8527a891e2    624
client_4e07408562    188
client_434c9b5ae5     42
client_8b940be7fb     19
Name: count, dtype: int64

False negatives by client:
client_id
client_4e07408562    924
client_e629fa6598    328
client_8527a891e2     48
Name: count, dtype: int64


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [40]:
# This cell is for CODE (numbers, a query, a check).
## 4. Claim rewrite

### Original claim

"Our model accurately predicts which pages are going to decline."

### Safer claim

"On the evaluated grouped-client split, the model measured directional performance in identifying pages associated with the observed declining label. The predictions may provide decision-support for prioritizing pages for further review, but they should not be interpreted as definitive predictions of future page decline or as evidence that acting on the predictions will improve SEO performance."
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


'On the evaluated grouped-client split, the model measured directional performance in identifying pages associated with the observed declining label. The predictions may provide decision-support for prioritizing pages for further review, but they should not be interpreted as definitive predictions of future page decline or as evidence that acting on the predictions will improve SEO performance.'

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit my repo URL on the card. Done.